# Finding Root Causes with NVIDIA Nsight Systems
---

In the previous notebook PyTorch Profiler told us that the backward pass was much slower than expected — but not why.  In this notebook we will use NVIDIA Nsight Systems (`nsys`) to look at the actual GPU timeline and find the root cause.

Like the PyTorch Profiler notebook, this is **hands-on**: you will add NVTX annotations yourself, observe what they reveal in the timeline, identify the bug, and fix it.  At each step there is a link to a reference solution if you get stuck.

## What Is Nsight Systems?

Nsight Systems is NVIDIA's system-wide performance analysis tool.  Where PyTorch Profiler shows you a step-level breakdown (DataLoader: 8 ms, Forward: 10 ms, Backward: 48 ms), Nsight Systems shows you the *continuous timeline* of everything happening on both the CPU and GPU — every CUDA kernel launch, every host↔device memory transfer, every synchronisation point.

This makes two classes of problem immediately visible that are completely invisible in a step-level view:

1. **GPU idle gaps** — periods where the GPU is doing nothing while the CPU runs, indicating an unexpected host↔device synchronisation.
2. **Serialised pipelines** — situations where kernels that should run concurrently are being forced to run one at a time.

### Running `nsys`

The basic command wraps your Python script:

```bash
nsys profile \
    --trace=cuda,nvtx,osrt \
    --output /workspace/reports/my_run \
    --force-overwrite true \
    python my_script.py
```

**Key flags:**

| Flag | Purpose |
|---|---|
| `--trace=cuda,nvtx,osrt` | Record CUDA kernels, NVTX ranges, and OS runtime events |
| `--output` | Path for the `.nsys-rep` output file (no extension needed) |
| `--force-overwrite true` | Overwrite any existing file at that path |

The output `.nsys-rep` file can be opened in the Nsight Systems desktop GUI (download from developer.nvidia.com/nsight-systems) or, if installed, viewed directly in JupyterLab via the Nsight JupyterLab extension.

### Limiting the Capture Window

Full traces of a 60-step training run are large and slow to open.  For diagnosis, we typically use warmup steps and capture only a handful of steps.  Our [`train_v2_nvtx.py`](../source_code/intro/train_v2_nvtx.py) script runs 60 steps with 5 warmup; to keep the file manageable we limit the capture with `--duration`:

```bash
nsys profile --duration 10 ...   # capture for 10 seconds then stop
```

Alternatively, you can embed `torch.cuda.cudart().cudaProfilerStart()` / `cudaProfilerStop()` in your script and use `--capture-range=cudaProfilerApi` to profile only a specific region.  For our short scripts the full run is fine.

## Step 1: Run nsys Without Any Annotations

Let's start by running nsys on [`train_v2.py`](../source_code/intro/train_v2.py) — the plain AMP script with no annotations.  This is what a raw nsys trace looks like before we add any labelling.

In [ ]:
!nsys profile \
    --trace=cuda,osrt \
    --output /workspace/reports/v2_plain \
    --force-overwrite true \
    python ../source_code/intro/train_v2.py

Open `v2_plain.nsys-rep` in the Nsight Systems GUI.

You will see rows of CUDA kernels in the GPU timeline — hundreds of them per step, named things like `volta_sgemm_128x64_tn`, `cudnn::...`, `elementwise_kernel`.  The timeline is dense and the kernel names are not informative without deep CUDA knowledge.

You can see that something seems off — there are gaps, and the backward phase looks different from the forward — but it is very hard to tell where one training section ends and another begins, or which part of the code produced which kernels.

This is the first lesson of nsys: **a raw trace is a sea of kernels.  You need annotations to navigate it.**

---
## Exercise: Add NVTX Range Annotations

NVTX (NVIDIA Tools Extension) is a lightweight annotation library.  You push a named range at the start of a code section and pop it at the end.  These ranges appear as coloured bands in the nsys timeline, directly above the GPU kernel rows, so you can immediately see which kernels belong to which section of your code.

```python
from torch.cuda import nvtx

# At the start of a section:
nvtx.range_push("forward")

# ... the forward pass ...

# At the end:
nvtx.range_pop()
```

NVTX ranges have essentially zero overhead when not profiling — nsys intercepts them only when actively recording.

### Your Task

Open [`train_v2.py`](../source_code/intro/train_v2.py) and add NVTX ranges wrapping each section of the training step:

1. `data_load` — around `next(loader_iter)`
2. `h2d` — around the `.to(device)` calls
3. `forward` — around the model forward pass and loss computation
4. `backward` — around `zero_grad` and `backward()`
5. `optimizer` — around `scaler.step()` and `scaler.update()`
6. `step` — around the entire step (outer range)

Here is a skeleton:

```python
for step in range(NUM_ITERS):

    nvtx.range_push("step")

    nvtx.range_push("data_load")
    x, y = next(loader_iter)
    nvtx.range_pop()

    nvtx.range_push("h2d")
    x = x.to(device, non_blocking=True)
    y = y.to(device, non_blocking=True)
    nvtx.range_pop()

    nvtx.range_push("forward")
    with torch.autocast("cuda", dtype=torch.float16):
        logits = model(x)
        loss = loss_fn(logits, y)
    nvtx.range_pop()

    nvtx.range_push("backward")
    optimizer.zero_grad(set_to_none=True)
    scaler.scale(loss).backward()
    nvtx.range_pop()

    nvtx.range_push("optimizer")
    scaler.step(optimizer)
    scaler.update()
    nvtx.range_pop()

    nvtx.range_pop()  # step
```

Five ranges per step: `data_load`, `h2d`, `forward`, `backward`, `optimizer`.  Simple, readable, immediately useful.

**Stuck?** See the reference solution: [`train_v2_nvtx.py`](../source_code/intro/train_v2_nvtx.py)

Once you have added NVTX ranges (or copied from the reference solution), run nsys with your annotated script:

In [ ]:
!nsys profile \
    --trace=cuda,nvtx,osrt \
    --output /workspace/reports/v2_nvtx \
    --force-overwrite true \
    python ../source_code/intro/train_v2_nvtx.py

Open `v2_nvtx.nsys-rep` in the Nsight Systems GUI.  You will now see the NVTX ranges as coloured bands.  Zoom in on a single `step` range.

**What you should see in the `forward` range:**  
A solid block of GPU kernels — convolutions and matrix multiplications running back-to-back, GPU fully occupied.  This looks healthy.

**What you should see in the `backward` range:**  
Something very different.  Instead of a solid block of kernels, you see a *sawtooth pattern*:

```
CPU  ─────────────────────────────────────────────────────────
      ▼sync ▼sync ▼sync ▼sync ▼sync ▼sync ▼sync ... (~60 times)
GPU  ▌▌ ░ ▌▌ ░ ▌▌ ░ ▌▌ ░ ▌▌ ░ ▌▌ ░ ▌▌ ░ ...
     └─┘ └─┘ └─┘ └─┘ └─┘ └─┘ └─┘
    kern gap kern gap kern gap kern
```

Each tiny block is the gradient computation for one layer.  Each gap is the GPU sitting idle while the CPU does something.  This pattern repeats **once per layer in the backward pass** — roughly 60 times per step for ResNet18.

This is a CPU↔GPU synchronisation happening after every single backward operation.  Every gradient kernel fires, then the GPU waits for the CPU to do some work, then the next gradient kernel fires.  The backward pass is completely serialised.

**The question is: what is causing that synchronisation?**

---
## Exercise: Identify and Fix the Bug

Look at what the CPU is doing during those gaps.  In the nsys CPU timeline, zoom into one of the idle gaps.  You will see the CPU is running Python code related to gradient checking.

Now look at [`train_v2.py`](../source_code/intro/train_v2.py) (or [`train_v2_nvtx.py`](../source_code/intro/train_v2_nvtx.py)) carefully:

```python
def train():
    torch.manual_seed(SEED)
    torch.autograd.set_detect_anomaly(True)  # left from debugging a NaN issue
    ...
```

**There it is.**

`torch.autograd.set_detect_anomaly(True)` is a debugging flag that installs a hook in the autograd engine.  After *every backward operation*, it:

1. Receives the gradient tensor on CPU
2. Runs `gradient.isnan().any()` to check for NaN or Inf values
3. Raises an exception if found

Step 2 requires the CPU to read the gradient value — which means the GPU must finish computing that gradient and transfer it to the CPU before the next backward operation can begin.  This is a host↔device synchronisation after *every single layer*, which is exactly the sawtooth pattern we are seeing.

With ResNet18's ~60 backward operations per step, that is 60 forced synchronisations where there would otherwise be zero.  The backward pass that should run as a continuous stream of GPU kernels is being chopped into 60 individual serialised pieces.

This flag is invaluable for debugging NaN/Inf gradient issues.  It is devastating for performance when left on by accident.

### Your Task

Remove (or comment out) the `set_detect_anomaly(True)` line in your script:

```python
def train():
    torch.manual_seed(SEED)
    # torch.autograd.set_detect_anomaly(True)  ← removed
    ...
```

**Stuck?** See the reference solution: [`train_v2_fixed.py`](../source_code/intro/train_v2_fixed.py)

Run your fixed script (or the reference solution) to verify the fix:

In [ ]:
!python ../source_code/intro/train_v2_fixed.py

**Expected output:**

```
steps timed: 55  mean step: ~47 ms  throughput: ~5450 img/s
```

| Script | Mean step | Throughput | vs FP32 baseline |
|---|---|---|---|
| [`train_v1_fixed.py`](../source_code/intro/train_v1_fixed.py) (FP32) | ~94 ms | ~2727 img/s | 1× |
| [`train_v2.py`](../source_code/intro/train_v2.py) (AMP + bug) | ~66 ms | ~3880 img/s | ~1.4× |
| [`train_v2_fixed.py`](../source_code/intro/train_v2_fixed.py) (AMP) | ~47 ms | ~5450 img/s | **~2×** |

The full ~2× AMP speedup is now visible.  Removing one debug flag recovered the ~27 % of throughput that the anomaly detection was consuming.

## Bonus: Auto-Generated NVTX with `--pytorch=autograd-nvtx`

Writing manual NVTX ranges gives you exactly the annotations you choose.  But sometimes you want to see *every* PyTorch operation annotated, not just the sections you labelled — for example, to see which specific layer inside `backward` is associated with each sync gap.

nsys 2026.1 supports a `--pytorch` flag that automatically instruments your script with per-operation NVTX ranges without any code changes:

```bash
nsys profile \
    --trace=cuda,nvtx,osrt \
    --pytorch=autograd-nvtx \
    --output /workspace/reports/v2_auto_nvtx \
    --force-overwrite true \
    python ../source_code/intro/train_v2.py
```

`--pytorch=autograd-nvtx` calls `torch.autograd.profiler.emit_nvtx(record_shapes=False)` automatically when PyTorch is imported — no changes to the script at all.

Let's run it:

In [ ]:
!nsys profile \
    --trace=cuda,nvtx,osrt \
    --pytorch=autograd-nvtx \
    --output /workspace/reports/v2_auto_nvtx \
    --force-overwrite true \
    python ../source_code/intro/train_v2.py

Open `v2_auto_nvtx.nsys-rep` in the GUI.

The NVTX row is now very dense — every individual PyTorch operation has its own range: `aten::conv2d`, `aten::batch_norm`, `aten::relu`, `CudnnConvolutionBackward`, and so on.  You can zoom in on the sawtooth gaps and see exactly which layer's backward operation each one corresponds to.

**Manual vs Auto NVTX — which to use?**

| | Manual NVTX | Auto NVTX (`--pytorch=autograd-nvtx`) |
|---|---|---|
| **Setup** | Add `range_push/pop` to code | Zero code changes |
| **Granularity** | You choose the sections | Every PyTorch op |
| **Timeline readability** | Clean, high-level | Dense, detailed |
| **Best for** | Understanding step structure | Drilling into a specific section |
| **Overhead** | Minimal | Moderate (many more ranges) |

A common workflow: start with manual NVTX to find *which section* has the problem (e.g., "backward is choppy"), then use `--pytorch=autograd-nvtx` to see *which operation within that section* is the culprit.

## Summary

Here is the full journey from the first slow measurement to the fixed result:

| Step | Tool | Finding |
|---|---|---|
| Timing | `time.perf_counter` | ~260 ms/step — too slow |
| Profiler | `torch.profiler` | DataLoader = 90% of step |
| Fix 1 | Code change | `num_workers=4`, `pin_memory=True` → 2.8× |
| Add AMP | Code change | Expected ~2×, got only ~1.4× |
| Profiler | `torch.profiler` | Backward section is slow — but why? |
| nsys (plain) | `nsys` | Sea of kernels, hard to navigate |
| nsys + NVTX | `nsys --trace=nvtx` | Sawtooth in backward: GPU idle after every layer |
| Root cause | Code inspection | `set_detect_anomaly(True)` left on |
| Fix 2 | Remove one line | Full 2× AMP speedup restored |

The key lesson: PyTorch Profiler and Nsight Systems answer *different* questions.  Profiler tells you **which section** is slow.  Nsight Systems tells you **why** — by showing you what the GPU and CPU are actually doing, and when they are waiting for each other.

## <center><div style="text-align:center; color:#FF0000; border:3px solid red; height:80px;"><b><br/>[Next Notebook — Nsight Systems In Depth](nsys-intermediate.ipynb)</b></div></center>

---

## Links and Resources

- [NVIDIA Nsight Systems](https://developer.nvidia.com/nsight-systems)
- [Nsight Systems User Guide](https://docs.nvidia.com/nsight-systems/UserGuide/index.html)
- [NVTX documentation](https://nvtx.readthedocs.io/en/latest/)
- [torch.cuda.nvtx API](https://pytorch.org/docs/stable/cuda.html#torch.cuda.nvtx.range_push)
- [torch.autograd.profiler.emit_nvtx](https://pytorch.org/docs/stable/autograd.html#torch.autograd.profiler.emit_nvtx)

---

## Licensing

Copyright © 2026 OpenACC-Standard.org. This material is released by OpenACC-Standard.org, in collaboration with NVIDIA Corporation, under the Creative Commons Attribution 4.0 International (CC BY 4.0).